In [1]:
"""
Retail Demand Forecasting — Improved / Interview-Defensible Modeling Pipeline
============================================================================
M5 (Walmart) dataset. Predicts daily units sold per item-store-day.


"""

import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error

print("xgboost version:", xgb.__version__)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



SUBSET_SIZE = 300


# ---------------------------------------------------------------------------
# 1. LOAD CLEANED DATA  (output of the data-exploration step: melt + merges)
# ---------------------------------------------------------------------------
df = pd.read_csv("../data/cleaned_sales.csv")
print("Loaded:", df.shape)

if SUBSET_SIZE is not None:
    all_ids = df["id"].unique()
    rng = np.random.default_rng(42)                 # seed = reproducible
    keep = rng.choice(all_ids, size=SUBSET_SIZE, replace=False)
    df = df[df["id"].isin(keep)].copy()
    print(f"Using random subset of {SUBSET_SIZE} products -> {df.shape}")

df["date"] = pd.to_datetime(df["date"])
# 'd_123' -> 123 as a real integer day index (better than label-encoding it)
df["d"] = df["d"].str.replace("d_", "", regex=False).astype(int)
df = df.sort_values(["id", "date"]).reset_index(drop=True)


xgboost version: 2.1.4
Loaded: (58327370, 22)
Using random subset of 300 products -> (573900, 22)


In [2]:
# ---------------------------------------------------------------------------
# 2. FEATURE ENGINEERING  
# ---------------------------------------------------------------------------
df["day_of_week"] = df["wday"]
df["is_weekend"]  = df["wday"].isin([1, 7]).astype(int)          # Sat=1, Sun=7 in M5
df["is_holiday"]  = ((df["event_type_1"] != "No Event") |
                     (df["event_type_2"] != "No Event")).astype(int)

grp = df.groupby("id")

# lag features: sales N days ago (shift keeps it strictly in the past)
df["lag_7"]  = grp["sales"].shift(7)
df["lag_28"] = grp["sales"].shift(28)

# rolling features: .shift(1) BEFORE .rolling() so the window ends yesterday,
# never including today's sales (this is what prevents target leakage)
df["rolling_mean_7"]  = grp["sales"].transform(lambda s: s.shift(1).rolling(7).mean())
df["rolling_mean_28"] = grp["sales"].transform(lambda s: s.shift(1).rolling(28).mean())
df["rolling_std_28"]  = grp["sales"].transform(lambda s: s.shift(1).rolling(28).std())

# price signals
df["price_lag_7"]    = grp["sell_price"].shift(7)
df["price_change_7"] = df["sell_price"] - df["price_lag_7"]

print("Feature engineering done.")

feature_cols = [
    "d", "wday", "month", "year",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI",
    "day_of_week", "is_weekend", "is_holiday",
    "sell_price", "price_lag_7", "price_change_7",
    "lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28", "rolling_std_28",
]
cat_cols = ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]
TARGET = "sales"



Feature engineering done.


In [3]:
# ---------------------------------------------------------------------------
# 3. CHRONOLOGICAL 3-WAY SPLIT  (M5 horizon = 28 days)
#      TRAIN = older | VALID = 28 days before test | TEST = final 28 days
# ---------------------------------------------------------------------------
max_date    = df["date"].max()
test_start  = max_date - pd.Timedelta(days=27)     # last 28 days
valid_start = test_start - pd.Timedelta(days=28)   # 28 days before that

train = df[df["date"] <  valid_start].copy()
valid = df[(df["date"] >= valid_start) & (df["date"] < test_start)].copy()
test  = df[df["date"] >= test_start].copy()
print(f"Train: {train.shape[0]:,}  Valid: {valid.shape[0]:,}  Test: {test.shape[0]:,}")


# ---------------------------------------------------------------------------
# 4. ENCODE CATEGORICALS  (fit on TRAIN only; map unseen categories to -1)
# ---------------------------------------------------------------------------
encoders = {}
for col in cat_cols:
    mapping = {v: i for i, v in enumerate(train[col].astype(str).unique())}
    encoders[col] = mapping
    for part in (train, valid, test):
        part[col] = part[col].astype(str).map(mapping).fillna(-1).astype(int)


# ---------------------------------------------------------------------------
# 5. IMPUTE MISSING VALUES  (medians learned on TRAIN, applied everywhere)
# ---------------------------------------------------------------------------
X_train, y_train = train[feature_cols].copy(), train[TARGET]
X_valid, y_valid = valid[feature_cols].copy(), valid[TARGET]
X_test,  y_test  = test[feature_cols].copy(),  test[TARGET]

medians = X_train.median()          # learned on train only -> no leakage
X_train = X_train.fillna(medians)
X_valid = X_valid.fillna(medians)
X_test  = X_test.fillna(medians)


# ---------------------------------------------------------------------------
# 6. NAIVE BASELINE  (predict "same as 7 days ago") — the bar to beat
# ---------------------------------------------------------------------------
naive_pred = X_test["lag_7"]
print(f"Naive baseline (lag_7) TEST RMSE: {rmse(y_test, naive_pred):.4f}")


Train: 557,100  Valid: 8,400  Test: 8,400
Naive baseline (lag_7) TEST RMSE: 2.6128


In [4]:

# ---------------------------------------------------------------------------
# 7. HYPERPARAMETER TUNING  
# ---------------------------------------------------------------------------
sample = train.sort_values("date")               # sort by time for TimeSeriesSplit
Xs, ys = sample[feature_cols].fillna(medians), sample[TARGET]
if len(Xs) > 300_000:                            # cap the tuning sample
    Xs, ys = Xs.iloc[-300_000:], ys.iloc[-300_000:]

param_dist = {
    "n_estimators":     [200, 300],
    "max_depth":        [4, 6],
    "learning_rate":    [0.05, 0.1],
    "subsample":        [0.8],
    "colsample_bytree": [0.7, 0.8],
    "min_child_weight": [1, 3],
    "gamma":            [0, 0.1],
}

search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(objective="reg:squarederror",
                               tree_method="hist", random_state=42,
                               n_jobs=1),          # model uses 1 thread
    param_distributions=param_dist,
    n_iter=8,                                      # fewer candidates
    cv=TimeSeriesSplit(n_splits=3),                # time-aware, not random KFold
    scoring="neg_root_mean_squared_error",
    verbose=2,
    random_state=42,
    n_jobs=1,                                      # KEY: no parallel data copies -> no OOM
)
search.fit(Xs, ys)
best_params = search.best_params_
print("Best params:", best_params)



Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.05, max_depth=6, min_child_weight=1, n_estimators=200, subsample=0.8; total time=   0.9s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.05, max_depth=6, min_child_weight=1, n_estimators=200, subsample=0.8; total time=   1.6s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.05, max_depth=6, min_child_weight=1, n_estimators=200, subsample=0.8; total time=   2.2s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.1, max_depth=4, min_child_weight=3, n_estimators=200, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.1, max_depth=4, min_child_weight=3, n_estimators=200, subsample=0.8; total time=   1.2s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.1, max_depth=4, min_child_weight=3, n_estimators=200, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, m

In [5]:
import xgboost as xgb
print(xgb.__version__)

2.1.4


In [6]:
# ---------------------------------------------------------------------------
# 8. FINAL MODEL  
# ---------------------------------------------------------------------------
final_model = xgb.XGBRegressor(
    **best_params,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=25,       
)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],  # the set early stopping watches
    verbose=50,
)

[0]	validation_0-rmse:4.34573
[50]	validation_0-rmse:1.92572
[100]	validation_0-rmse:1.89637
[124]	validation_0-rmse:1.90180


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=25,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0.1, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [7]:
# ---------------------------------------------------------------------------
# 9. EVALUATE ON THE UNTOUCHED TEST SET  
# ---------------------------------------------------------------------------
test_pred  = final_model.predict(X_test)
model_rmse = rmse(y_test, test_pred)
naive_rmse = rmse(y_test, naive_pred)
print(f"\nFINAL TEST RMSE (model): {model_rmse:.4f}")
print(f"Naive baseline  RMSE   : {naive_rmse:.4f}")
print(f"Improvement over naive : {(1 - model_rmse / naive_rmse) * 100:.1f}%")


FINAL TEST RMSE (model): 1.9236
Naive baseline  RMSE   : 2.6128
Improvement over naive : 26.4%


In [8]:
# ---------------------------------------------------------------------------
# 10. SAVE MODEL + ENCODERS + MEDIANS  
# ---------------------------------------------------------------------------
final_model.save_model("final_xgb_model.json")
with open("label_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
with open("feature_medians.pkl", "wb") as f:
    pickle.dump(medians, f)
print("Saved model, encoders, and medians.")

Saved model, encoders, and medians.


In [9]:
# ===========================================================================
# 11. TREND ANALYSIS + ANOMALY DETECTION  
# ===========================================================================
# Trend: total daily sales with a 28-day rolling average (level + direction)
daily = df.groupby("date")["sales"].sum().reset_index()
daily["trend_28d"] = daily["sales"].rolling(28).mean()

# Anomaly detection: per-series rolling z-score; |z| > 3 = unusual demand
roll_mean = grp["sales"].transform(lambda s: s.shift(1).rolling(28).mean())
roll_std  = grp["sales"].transform(lambda s: s.shift(1).rolling(28).std())
df["z_score"] = (df["sales"] - roll_mean) / roll_std
df["anomaly"] = (df["z_score"].abs() > 3).astype(int)
n_anom = int(df["anomaly"].sum())
print(f"\nFlagged {n_anom:,} anomalous item-days "
      f"({n_anom / len(df) * 100:.2f}% of records) via rolling z-score.")

# Most volatile products (highest demand std) — operational risk signal
volatility = (df.groupby("item_id")["sales"].std()
                .sort_values(ascending=False).head(10))
print("\nTop 10 most volatile products (demand std):")
print(volatility)


Flagged 12,146 anomalous item-days (2.12% of records) via rolling z-score.

Top 10 most volatile products (demand std):
item_id
FOODS_3_090      17.902874
FOODS_3_586      17.782236
FOODS_3_541      15.538655
FOODS_3_635      12.565217
FOODS_3_816      11.910541
FOODS_3_086       7.601873
FOODS_3_227       7.582175
HOBBIES_1_178     6.262369
HOBBIES_1_080     5.429430
HOBBIES_1_319     5.223557
Name: sales, dtype: float64
